In [1]:
import os
import copy
import random
import time
from operator import itemgetter
from collections import defaultdict, deque

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.autograd import Variable
import torch.optim as optim
import gym
from gym.spaces import Box, Discrete
import matplotlib.pyplot as plt
from IPython import display

In [6]:
class GomokuEnv:
    def __init__(self, width=15, height=15, n_in_row=5):
        self.width = width
        self.height = height
        self.n_in_row = n_in_row
        self.players = [1, 2]  # Player 1 (Black) and Player 2 (White)
        self.reset()
        
    def reset(self):
        self.board = np.zeros((self.height, self.width), dtype=int)
        self.availables = list(range(self.width * self.height))
        self.current_player = self.players[0]
        self.last_move = -1
        self.states = {}
        
    def move_to_location(self, move):
        h = move // self.width
        w = move % self.width
        return [h, w]
    
    def location_to_move(self, location):
        if len(location) != 2:
            return -1
        h, w = location
        move = h * self.width + w
        return move
    
    def current_state(self):
        """Return 4-channel board state"""
        state = np.zeros((4, self.height, self.width), dtype=np.float32)
        
        # Channel 0: Current player's stones
        # Channel 1: Opponent's stones
        # Channel 2: Last move position
        # Channel 3: Turn indicator
        
        for move, player in self.states.items():
            h, w = self.move_to_location(move)
            if player == self.current_player:
                state[0][h][w] = 1
            else:
                state[1][h][w] = 1
        
        if self.last_move != -1:
            h, w = self.move_to_location(self.last_move)
            state[2][h][w] = 1
        
        state[3][:,:] = 1 if len(self.states) % 2 == 0 else 0
        return state
    
    def has_winner(self):
        moved = list(set(range(self.width*self.height)) - set(self.availables))
        if len(moved) < self.n_in_row *2 -1:
            return False, -1
        
        for m in moved:
            h = m // self.width
            w = m % self.width
            player = self.states[m]
            
            # Check horizontal
            if w <= self.width - self.n_in_row:
                if all(self.states.get(h*self.width+w+i, -1) == player for i in range(self.n_in_row)):
                    return True, player
                    
            # Check vertical
            if h <= self.height - self.n_in_row:
                if all(self.states.get((h+i)*self.width+w, -1) == player for i in range(self.n_in_row)):
                    return True, player
                    
            # Check diagonal
            if w <= self.width - self.n_in_row and h <= self.height - self.n_in_row:
                if all(self.states.get((h+i)*self.width+w+i, -1) == player for i in range(self.n_in_row)):
                    return True, player
                    
            # Check anti-diagonal
            if w >= self.n_in_row -1 and h <= self.height - self.n_in_row:
                if all(self.states.get((h+i)*self.width+w-i, -1) == player for i in range(self.n_in_row)):
                    return True, player
        return False, -1
    
    def game_ended(self):
        return self.has_winner()[0] or not self.availables

class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        # Modified to match state_dict keys
        self.conv1 = nn.Conv2d(4, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.act_conv1 = nn.Conv2d(128, 4, kernel_size=1)
        self.act_fc1 = nn.Linear(4*15*15, 225)
        self.val_conv1 = nn.Conv2d(128, 2, kernel_size=1)
        self.val_fc1 = nn.Linear(2*15*15, 64)
        self.val_fc2 = nn.Linear(64, 1)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = F.relu(self.conv3(x))
        
        x_act = F.relu(self.act_conv1(x))
        x_act = x_act.view(-1, 4*15*15)
        x_act = F.log_softmax(self.act_fc1(x_act), dim=1)
        
        x_val = F.relu(self.val_conv1(x))
        x_val = x_val.view(-1, 2*15*15)
        x_val = F.relu(self.val_fc1(x_val))
        x_val = torch.tanh(self.val_fc2(x_val))
        
        return x_act, x_val

class PolicyValueNet:
    def __init__(self, model_path):
        # Instantiate model
        self.model = Net()
        
        # Load trained weights
        self.model.load_state_dict(torch.load(model_path, map_location=torch.device('cpu')))
        self.model.eval()

    def policy_value_fn(self, board):
        """Return policy probabilities without exploration"""
        state = board.current_state()
        state_tensor = torch.FloatTensor(state).unsqueeze(0)
        
        with torch.no_grad():
            log_act_probs, value = self.model(state_tensor)
            
        act_probs = np.exp(log_act_probs.numpy().flatten())
        legal_moves = board.availables
        
        # Return raw probabilities without normalization
        probs = [(move, act_probs[move]) for move in legal_moves]
        return probs, value.item()


class MCTSPlayer:
    def __init__(self, policy_value_net, n_playout=200):
        self.policy_value_net = policy_value_net
        self.n_playout = n_playout
        
    def get_action(self, board):
        """Always select the move with highest probability"""
        move_probs = self.policy_value_net.policy_value_fn(board)[0]
        
        if not move_probs:
            return np.random.choice(board.availables)
            
        moves, probs = zip(*move_probs)
        
        # Select move with maximum probability
        max_prob_index = np.argmax(probs)
        return moves[max_prob_index]

    def policy_value_fn(self, board):
        """Get raw probabilities without exploration"""
        probs, value = self.policy_value_net.policy_value_fn(board)
        
        # Convert to list of (move, probability) tuples
        legal_probs = [(move, prob) for move, prob in probs]
        return legal_probs, value

class GameVisualizer:
    def __init__(self, board_size=15):
        self.board_size = board_size
        self.fig, self.ax = plt.subplots(figsize=(8,8))
        self.cmap = plt.cm.Blues  # Changed to blue colormap
        self.norm = plt.Normalize(vmin=0, vmax=1)
        self.fixed_color = '#1F77B4'
        self.fig.patch.set_facecolor('#e4ce9f')  # Background color for the figure
        self.ax.set_facecolor('#e4ce9f')
        
    def update(self, board, policy_probs=None):
        self.ax.clear()
        self.ax.set_facecolor('#e4ce9f')
        
        # Create board grid
        for i in range(self.board_size):
            self.ax.plot([i, i], [0, self.board_size-1], 'k')
            self.ax.plot([0, self.board_size-1], [i, i], 'k')

        # Add stone first
        for move, player in board.states.items():
            h, w = board.move_to_location(move)
            color = 'black' if player == 1 else 'white'
            edge = 'black' if player == 2 else 'white'  # Black border for white stones
            self.ax.scatter(w, h, s=600,
                           c=color,
                           edgecolors=edge,
                           linewidths=1.5,
                           zorder=2)
        
        # Add policy shades on empty spots
        if policy_probs is not None:
            for move, prob in policy_probs:
                h, w = board.move_to_location(move)
                alpha = prob * 0.9 + 0.1
                self.ax.scatter(w, h, s=600,
                              c=self.fixed_color, 
                              edgecolors='none', 
                              alpha=alpha,
                              zorder=1)  # Lower zorder to stay under stones
        
        # Configure axis limits
        self.ax.set_xlim(-0.5, self.board_size-0.5)
        self.ax.set_ylim(-0.5, self.board_size-0.5)
        self.ax.set_xticks([])
        self.ax.set_yticks([])
        
        # Display and pause if showing policy
        display.display(self.fig)
        display.clear_output(wait=True)
        if policy_probs is not None:
            time.sleep(1)



def play_game(human_player=None):
    env = GomokuEnv()
    net = PolicyValueNet("/kaggle/input/1000model/pytorch/default/1/1000.pt")  # Load your trained model
    mcts_player = MCTSPlayer(net)
    visualizer = GameVisualizer()

    while not env.game_ended():
        visualizer.update(env, None)
        
        if human_player == env.current_player:
            print("Current board:")
            visualizer.update(env)  # Show current board state
            print("Your turn! Enter move as 'row column' (1-15 1-15)")
            while True:
                try:
                    visualizer.update(env, None)
                    move_input = input("Your move: ")
                    h, w = map(int, move_input.strip().split())
                    move = env.location_to_move([h-1, w-1])
                    
                    if move in env.availables:
                        break
                    print("Invalid move! Try again.")
                except (ValueError, IndexError):
                    print("Invalid input! Use format: '3 5' for row 3, column 5")
        else:
            # AI player's turn
            move = mcts_player.get_action(env)
            policy_probs = net.policy_value_fn(env)[0]
            visualizer.update(env, policy_probs)
        
        # Execute move
        h, w = env.move_to_location(move)
        env.board[h][w] = env.current_player
        env.availables.remove(move)
        env.states[move] = env.current_player
        env.current_player = 2 if env.current_player == 1 else 1
    
    visualizer.update(env, None)
    ended, winner = env.has_winner()
    if ended:
        print(f"Player {winner} wins!")
        visualizer.update(env, None)
    else:
        print("Game ended in a draw!")
        visualizer.update(env, None)

In [ ]:
# To watch AI vs AI:
play_game()

In [ ]:
## To human play against AI (can change to white by human_player=2):
play_game(human_player=1)